# 04 · Minería no supervisada

Tres análisis, con las funciones de `src/mineria` (probadas en
`tests/test_mineria.py`):

1. **K-means sobre la forma del día**: cada día dividido por su media, para que
   la tendencia no decida los grupos. El calendario NO entra al algoritmo; se
   usa después para leer los grupos.
2. **Reglas de asociación (Apriori)** entre grupo, tipo de día, nivel y clima.
3. **Sectores CIIU por perfil semanal**: qué subactividades se comportan como
   industria de proceso continuo.

Las figuras del informe salen de `python scripts/mineria_no_supervisada.py`.

In [ ]:
import sys, warnings
from pathlib import Path

# Sube desde la carpeta del notebook hasta la raiz del proyecto, la que contiene
# src/ingesta: funciona igual en notebooks/ que en notebooks/limpieza/<dataset>/.
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "ingesta").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

DATASETS = RAIZ / "datasets"
print("raiz     :", RAIZ)
print("datasets :", DATASETS, "->", "existe" if DATASETS.exists() else "FALTA")

In [ ]:
PROCESADO = RAIZ / "data" / "procesado"
panel = pd.read_parquet(PROCESADO / "panel_modelado.parquet")
print(f"{len(panel):,} filas x {panel.shape[1]} columnas · {panel.fecha_hora.min()} .. {panel.fecha_hora.max()}")

In [ ]:
from mineria.perfiles import perfiles_diarios, forma_y_nivel, atributos_dia
perfiles, registro = perfiles_diarios(panel)
forma, nivel = forma_y_nivel(perfiles)
atributos = atributos_dia(panel, perfiles.index)
registro

## Elección de k

Silueta como criterio, estabilidad (ARI entre semillas y submuestras) como freno.

In [ ]:
from mineria.clustering import evaluar_k, elegir_k, agrupar, describir, contingencia, v_cramer, dias_atipicos
evaluacion = evaluar_k(forma.to_numpy(), ks=range(2, 11))
k = elegir_k(evaluacion)
print("k elegido:", k)
evaluacion.round(3)

In [ ]:
resultado = agrupar(forma, k)
print(f"silueta {resultado.silueta:.3f} · ARI frente a Ward {resultado.ari_ward:.3f}")
describir(resultado, forma, nivel, atributos)

In [ ]:
tabla = contingencia(resultado, atributos)
print("V de Cramér grupo x tipo de día:", round(v_cramer(tabla), 3))
tabla

In [ ]:
fig, eje = plt.subplots(figsize=(12, 4))
resultado.centroides.T.plot(ax=eje, marker="o")
eje.set_xlabel("hora"); eje.set_ylabel("demanda / media del día"); eje.set_title("Centroides");

## Días que no se parecen a su grupo

In [ ]:
dias_atipicos(resultado, atributos, cuantil=0.99).head(15)

## Reglas de asociación

In [ ]:
from mineria.reglas import transacciones, reglas_asociacion, podar_redundantes, reglas_hacia
canasta = transacciones(atributos, resultado.etiquetas, nivel)
reglas = podar_redundantes(reglas_asociacion(canasta))
print(len(reglas), "reglas no redundantes")
reglas_hacia(reglas, "grupo=").head(10)

In [ ]:
reglas.head(15)

## Sectores CIIU por perfil semanal

Requiere `datasets/limpios/ciiu_limpio.parquet` (notebook `limpieza/xm_ciiu`).

In [ ]:
ruta_ciiu = DATASETS / "limpios" / "ciiu_limpio.parquet"
if ruta_ciiu.exists():
    from sklearn.cluster import KMeans
    from mineria.sectores import perfiles_sectoriales, describir_sectores, ordenar_por_tamano, mayores_por_grupo
    ciiu = pd.read_parquet(ruta_ciiu, columns=["timestamp", "activity", "subactivity", "valor", "origen_valor"])
    perf, energia, reg = perfiles_sectoriales(ciiu)
    del ciiu
    ev = evaluar_k(perf.to_numpy(), ks=range(2, 9))
    ks = elegir_k(ev, tamano_minimo=5)
    etiquetas = pd.Series(ordenar_por_tamano(KMeans(ks, n_init=20, random_state=42).fit_predict(perf.to_numpy())), index=perf.index)
    display(reg, describir_sectores(perf, energia, etiquetas), mayores_por_grupo(energia, etiquetas))
else:
    print("Falta", ruta_ciiu, "-> ejecuta notebooks/limpieza/xm_ciiu")